<a href="https://colab.research.google.com/github/netsetos/genai-engg-course-learners/blob/main/module-00-prerequisites/lesson-0.1-python-for-genai/notebooks/exercises-0_1.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lesson 0.1: Python for GenAI

*Module 0 · Prerequisites*

> You know Python basics. But GenAI engineering demands specific patterns: async for parallel API calls, generators for streaming, Pydantic for structured outputs, decorators for retry logic, proper error handling for unreliable APIs, and a production call pattern that combines everything. This is the Python no other GenAI course teaches.

`Prerequisite` · `8 Skills` · `All Runnable` · `90 min`

---

## About this notebook

This notebook contains the **7 runnable code examples** from the published lesson `lesson-0.1.html`. Each block is reproduced verbatim — every `#` comment from the source is preserved — and is preceded by a short markdown header that names the step, the file, and what the block demonstrates.

**How to use it:**

1. Run the **Setup** cells once (install + API keys).
2. Step through the lesson cells in order — each is independent enough to inspect on its own.
3. Read the *What just happened?* recap that follows each block (where the lesson provides one).


## Contents

1. Step 1 — Async / Await — 5x Faster API Calls — `async_demo.py`
2. Step 2 — Generators & Yield — Streaming LLM Tokens — `generators_demo.py`
3. Step 3 — Type Hints & Pydantic — Taming LLM Outputs — `pydantic_demo.py`
4. Step 4 — Decorators — Auto-Retry, Cache, Rate Limit — `decorators_demo.py`
5. Step 5 — Env Variables & JSON — Secure Keys, Clean Data — `env_and_json.py`
6. Step 6 — Error Handling & Resilience — LLM APIs Will Fail — `error_handling.py`
7. Step 7 — The Production LLM Call — All Skills Combined — `production_pattern.py`


## Setup

Run these cells once per environment. Skip any step you've already done.


In [ ]:
# Install Python dependencies used by this lesson.
# The -q flag keeps output quiet; remove it if you want full pip logs.
!pip install -q transformers pydantic python-dotenv


In [ ]:
# Load API keys from the environment.
# Before running the lesson cells, export each key in your shell, e.g.:
#   export GOOGLE_API_KEY=sk-...
# (Or replace the placeholder below with your real key for a quick local test.)
import os
os.environ.setdefault("GOOGLE_API_KEY", "YOUR_GOOGLE_API_KEY_HERE")


## Lesson code

Each section below corresponds to one code block from the published lesson.


### Step 1 · Async / Await — 5x Faster API Calls

LLM calls take 2-5 seconds each. Sequential = slow. Async = parallel.

**`async_demo.py`** · _Block 1 of 7_

ASYNC — Run 5 LLM calls in parallel — Sequential: 10 seconds. Async: 2 seconds.


In [ ]:
# =============================================
# ASYNC — Run 5 LLM calls in parallel
# Sequential: 10 seconds. Async: 2 seconds.
# =============================================

import asyncio, time

# Simulate an LLM API call (takes 2 seconds)
async def call_llm(prompt: str, call_id: int) -> str:
    print(f"  ⏳ Call {call_id} started: '{prompt[:30]}...'")
    await asyncio.sleep(2)  # simulates network wait
    return f"Response to: {prompt[:20]}"

# ── Sequential (slow) ──
async def sequential():
    start = time.time()
    prompts = ["What is AI?", "What is ML?", "What is DL?", "What is NLP?", "What is CV?"]
    results = []
    for i, p in enumerate(prompts):
        r = await call_llm(p, i+1)
        results.append(r)
    print(f"  Sequential: {time.time()-start:.1f}s for {len(results)} calls\n")

# ── Async (fast) ──
async def parallel():
    start = time.time()
    prompts = ["What is AI?", "What is ML?", "What is DL?", "What is NLP?", "What is CV?"]
    tasks = [call_llm(p, i+1) for i, p in enumerate(prompts)]
    results = await asyncio.gather(*tasks)  # ← magic line!
    print(f"  Async:      {time.time()-start:.1f}s for {len(results)} calls")

print("Sequential:")
asyncio.run(sequential())
print("Parallel:")
asyncio.run(parallel())


> **What just happened?** Sequential: 5 calls × 2 seconds = 10 seconds . Async: all 5 calls fire at once, wait for the slowest (2 seconds) = 2 seconds . The magic line is asyncio.gather(*tasks) — it runs all coroutines concurrently. You'll use this in Module 5 (multi-provider API calls) and Module 11 (parallel batch processing).


### Step 2 · Generators & Yield — Streaming LLM Tokens

LLMs generate one token at a time. Generators let you process each token as it arrives.

**`generators_demo.py`** · _Block 2 of 7_

GENERATORS — Stream tokens like an LLM — yield sends one item at a time instead of


In [ ]:
# =============================================
# GENERATORS — Stream tokens like an LLM
# yield sends one item at a time instead of
# waiting to build the entire response.
# =============================================

import time

# ── Normal function: waits until EVERYTHING is ready ──
def get_response_normal() -> str:
    words = ["The", "future", "of", "AI", "is", "bright"]
    result = []
    for word in words:
        time.sleep(0.3)  # simulate generation time
        result.append(word)
    return " ".join(result)  # user waits 1.8s, then sees everything

# ── Generator: streams word by word ──
def stream_response():
    words = ["The", "future", "of", "AI", "is", "bright"]
    for word in words:
        time.sleep(0.3)
        yield word  # ← sends each word IMMEDIATELY

# Normal: user sees nothing for 1.8s, then everything
print("Normal function:")
start = time.time()
result = get_response_normal()
print(f"  {result} (waited {time.time()-start:.1f}s)\n")

# Generator: user sees words appear one by one
print("Generator (streaming):")
start = time.time()
print("  ", end="")
for word in stream_response():
    print(word, end=" ", flush=True)  # appears immediately!
print(f"\n  (first word at 0.3s, total {time.time()-start:.1f}s)")


> **What just happened?** Normal function: user stares at a blank screen for 1.8 seconds, then everything appears at once. Generator with yield : user sees the first word in 0.3 seconds, then each word appears progressively. Same total time — but the perceived latency drops from 1.8s to 0.3s. This is exactly how ChatGPT's streaming works. You'll build this in Module 5 (streaming) and Module 10 (AI-powered UX).


### Step 3 · Type Hints & Pydantic — Taming LLM Outputs

LLMs return messy text. Pydantic forces it into clean, validated Python objects.

**`pydantic_demo.py`** · _Block 3 of 7_

PYDANTIC — Validate messy LLM output — LLM returns JSON text. Pydantic ensures it's


In [ ]:
# =============================================
# PYDANTIC — Validate messy LLM output
# LLM returns JSON text. Pydantic ensures it's
# actually valid and has the right types.
# =============================================

from pydantic import BaseModel, Field
from typing import Optional
import json

# Define WHAT the LLM output should look like
class MovieReview(BaseModel):
    title: str = Field(description="Movie title")
    rating: float = Field(ge=1, le=10, description="Rating 1-10")
    summary: str = Field(max_length=200)
    recommended: bool
    genre: Optional[str] = None

# Simulate LLM returning JSON text
llm_output = '{"title": "Jawan", "rating": 8.5, "summary": "Shah Rukh Khan delivers a powerful performance in this action thriller.", "recommended": true, "genre": "Action"}'

# Parse and validate — Pydantic catches errors automatically
review = MovieReview.model_validate_json(llm_output)
print(f"Title:       {review.title}")
print(f"Rating:      {review.rating}/10")
print(f"Recommended: {review.recommended}")
print(f"Type check:  rating is {type(review.rating).__name__} ✅")

# What if the LLM returns garbage?
print("\nBad LLM output:")
try:
    bad = MovieReview.model_validate_json('{"title": "X", "rating": 15, "summary": "ok", "recommended": "yes"}')
except Exception as e:
    print(f"  ❌ Pydantic caught it: {e}")
    print("  Rating 15 is > 10, 'yes' isn't a bool → rejected!")


> **What just happened?** Pydantic validated the LLM's JSON against a schema: title must be a string, rating must be 1-10, summary under 200 chars, recommended must be a bool. When the LLM returned "rating": 15 (out of range) and "recommended": "yes" (not a bool), Pydantic caught it instantly. You'll use this in Module 3 (structured output), Module 5 (API responses), and Module 8 (MCP tool schemas).


### Step 4 · Decorators — Auto-Retry, Cache, Rate Limit

LLM APIs fail. Decorators add retry logic, caching, and logging without changing your code.

**`decorators_demo.py`** · _Block 4 of 7_

DECORATORS — Add superpowers to any function — Auto-retry failed API calls. Cache results.


In [ ]:
# =============================================
# DECORATORS — Add superpowers to any function
# Auto-retry failed API calls. Cache results.
# Time execution. All without changing the function.
# =============================================

import time, functools, random

# ── Retry decorator: auto-retry on failure ──
def retry(max_attempts=3, delay=1):
    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            for attempt in range(1, max_attempts + 1):
                try:
                    return func(*args, **kwargs)
                except Exception as e:
                    print(f"  ⚠️ Attempt {attempt} failed: {e}")
                    if attempt < max_attempts:
                        time.sleep(delay)
            raise Exception(f"Failed after {max_attempts} attempts")
        return wrapper
    return decorator

# ── Timer decorator: measure execution time ──
def timer(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        start = time.time()
        result = func(*args, **kwargs)
        print(f"  ⏱️ {func.__name__} took {time.time()-start:.2f}s")
        return result
    return wrapper

# ── Use both decorators on an LLM call ──
@retry(max_attempts=3, delay=0.5)
@timer
def call_llm(prompt: str) -> str:
    # Simulate: 50% chance of failure (like a flaky API)
    if random.random() < 0.5:
        raise ConnectionError("API timeout")
    return f"Response to: {prompt}"

print("Calling flaky API with auto-retry:\n")
try:
    result = call_llm("What is Python?")
    print(f"  ✅ Got: {result}")
except:
    print("  ❌ All retries exhausted")


> **What just happened?** @retry automatically retries the function up to 3 times if it fails. @timer measures execution time. The function itself doesn't know about retrying or timing — the decorators add those superpowers externally. In production, you'll stack decorators: @retry @cache @rate_limit @log on every LLM call. Module 10 builds a full AI gateway using this pattern.


### Step 5 · Env Variables & JSON — Secure Keys, Clean Data

Never hardcode API keys. Always validate JSON from LLMs.

**`env_and_json.py`** · _Block 5 of 7_

ENV VARS — Never hardcode API keys! — JSON — Parse, validate, handle LLM output.


In [ ]:
# =============================================
# ENV VARS — Never hardcode API keys!
# JSON — Parse, validate, handle LLM output.
# =============================================

import os, json

# ── Environment variables ──
# Set in terminal: export GOOGLE_API_KEY="your-key-here"
# Or in .env file with python-dotenv

api_key = os.getenv("GOOGLE_API_KEY", "NOT_SET")
print(f"API Key: {api_key[:8]}... (first 8 chars only for safety)")

if api_key == "NOT_SET":
    print("⚠️ Set your key: export GOOGLE_API_KEY='your-key'")

# ── Safely parse JSON from LLM ──
# LLMs sometimes return JSON wrapped in ```json ... ```
def safe_parse_json(text: str) -> dict:
    """Parse JSON from LLM output, handling markdown fences."""
    # Strip markdown code fences
    cleaned = text.strip()
    if cleaned.startswith("```json"):
        cleaned = cleaned[7:]
    if cleaned.startswith("```"):
        cleaned = cleaned[3:]
    if cleaned.endswith("```"):
        cleaned = cleaned[:-3]
    
    try:
        return json.loads(cleaned.strip())
    except json.JSONDecodeError as e:
        print(f"❌ Invalid JSON: {e}")
        return {}

# Test with messy LLM output
messy_output = '```json\n{"name": "Gemini", "score": 9.5}\n```'
parsed = safe_parse_json(messy_output)
print(f"\nParsed: {parsed}")
print(f"Name:   {parsed.get('name')}")
print(f"Score:  {parsed.get('score')}")


> **What just happened?** os.getenv() reads API keys from environment variables — never hardcode keys in source code. If you push GOOGLE_API_KEY="abc123" to GitHub, bots will find it in minutes and run up your bill. The safe_parse_json() function handles a real production problem: LLMs often wrap JSON in markdown fences ( ```json...``` ). This function strips them before parsing. You'll use both patterns in every module of this course.


### Step 6 · Error Handling & Resilience — LLM APIs Will Fail

Rate limits, timeouts, server errors. Your app must handle all of them gracefully.

**`error_handling.py`** · _Block 6 of 7_

LLM ERROR HANDLING — The patterns that keep production apps alive


In [ ]:
# =============================================
# LLM ERROR HANDLING — The patterns that keep production apps alive
# =============================================

import time, random

# ── 1. Structured exception handling for LLM APIs ──
# Different errors need different responses:
#   429 Rate Limit  → wait and retry (exponential backoff)
#   500 Server Error → retry (might be temporary)
#   401 Auth Error  → FAIL FAST (don't retry — key is wrong)
#   Timeout         → retry with longer timeout

class RateLimitError(Exception): pass
class ServerError(Exception): pass
class AuthError(Exception): pass

def call_llm_unreliable(prompt: str) -> str:
    """Simulates an unreliable LLM API."""
    roll = random.random()
    if roll < 0.3: raise RateLimitError("429: Too Many Requests")
    if roll < 0.4: raise ServerError("500: Internal Server Error")
    if roll < 0.42: raise AuthError("401: Invalid API Key")
    time.sleep(0.5)
    return f"Response for: {prompt}"

# ── 2. Exponential backoff with jitter ──
def retry_with_backoff(func, prompt, max_retries=3):
    """Retry with exponential backoff. Fail fast on auth errors."""
    for attempt in range(max_retries):
        try:
            return func(prompt)
        except AuthError:
            print(f"  ❌ Auth error — FAIL FAST (don't retry)")
            raise  # Never retry auth errors!
        except RateLimitError:
            wait = (2 ** attempt) + random.uniform(0, 1)  # Jitter prevents thundering herd
            print(f"  ⏳ Rate limited. Retry {attempt+1}/{max_retries} in {wait:.1f}s")
            time.sleep(wait)
        except ServerError:
            print(f"  🔄 Server error. Retry {attempt+1}/{max_retries}")
            time.sleep(1)
    raise Exception(f"Failed after {max_retries} retries")

# ── 3. Model fallback chain ──
def call_with_fallback(prompt: str) -> str:
    """Try primary model, fall back to cheaper, then to cache."""
    models = ["gemini-2.0-flash", "gemini-2.0-flash-lite", "cached"]
    for model in models:
        try:
            print(f"  Trying {model}...")
            if model == "cached":
                return "[Cached fallback response]"
            return retry_with_backoff(call_llm_unreliable, prompt)
        except Exception as e:
            print(f"  {model} failed: {e}")
    return "Service temporarily unavailable"

# Test
print("Calling LLM with full error handling:\n")
result = call_with_fallback("What is GenAI?")
print(f"\nFinal result: {result}")
print(f"\nIn production: use 'tenacity' library instead of hand-rolled retry")
print(f"  pip install tenacity")
print(f"  @retry(wait=wait_exponential(min=1, max=60), stop=stop_after_attempt(3))")


> **What just happened?** Three resilience layers: (1) Structured error handling — different errors get different treatment. Auth errors fail fast (retrying a wrong key wastes time). Rate limits wait and retry. (2) Exponential backoff with jitter — wait 1s, 2s, 4s... plus random jitter so 1000 clients don't all retry at the same second. (3) Model fallback chain — if the primary model is down, try a cheaper one, then serve a cached response. This is how Netflix-scale GenAI apps stay up during API outages. In production, use the tenacity library instead of hand-rolled retry (Module 10).


### Step 7 · The Production LLM Call — All Skills Combined

Every skill in this lesson combined into one production-ready pattern.

**`production_pattern.py`** · _Block 7 of 7_

THE PRODUCTION LLM CALL — All 7 skills combined — This is exactly how Modules 5-11 make LLM calls


In [ ]:
# =============================================
# THE PRODUCTION LLM CALL — All 7 skills combined
# This is exactly how Modules 5-11 make LLM calls
# =============================================

import asyncio, os, json, time, functools
from pydantic import BaseModel, Field

# ── Skill 5: Env vars ──
API_KEY = os.getenv("GOOGLE_API_KEY", "demo-key")
assert API_KEY != "NOT_SET", "Set GOOGLE_API_KEY!"

# ── Skill 3+4: Pydantic validates LLM output ──
class LLMResponse(BaseModel):
    answer: str = Field(..., min_length=1)
    confidence: float = Field(..., ge=0, le=1)
    tokens: int = Field(..., ge=1)

# ── Skill 4: Decorators stack ──
def cache(func):
    _c = {}
    @functools.wraps(func)
    async def wrapper(*a):
        k = str(a)
        if k in _c: return _c[k]
        r = await func(*a); _c[k] = r; return r
    return wrapper

# ── Skill 1+6: Async + error handling + retry ──
@cache
async def call_llm(prompt: str) -> LLMResponse:
    for attempt in range(3):
        try:
            await asyncio.sleep(0.5)  # Simulated API call
            raw = {"answer": f"GenAI uses transformers for {prompt}",
                   "confidence": 0.92, "tokens": 45}
            return LLMResponse(**raw)  # Pydantic validates!
        except Exception:
            if attempt < 2: await asyncio.sleep(2 ** attempt)
            else: raise

# ── Skill 2: Generator for streaming ──
def stream(text, delay=0.02):
    for word in text.split():
        time.sleep(delay)
        yield word + " "

# ── Run the pipeline ──
async def main():
    prompts = ["RAG", "agents", "MCP", "RAG"]  # Note: RAG appears twice (cache test)
    
    # Parallel calls with async
    start = time.time()
    results = await asyncio.gather(*[call_llm(p) for p in prompts])
    print(f"4 calls in {time.time()-start:.1f}s (parallel + 1 cached)\n")
    
    # Stream each result
    for r in results:
        print(f"[{r.confidence:.0%} conf, {r.tokens} tokens] ", end="")
        for w in stream(r.answer): print(w, end="", flush=True)
        print()

await main()


> **What just happened?** Every skill combined: env vars load the API key. Pydantic validates the response. Decorators add caching. Async runs 4 calls in parallel (the duplicate "RAG" hits cache). Error handling retries on failure. Generators stream each word. This is the exact architecture you'll build in Module 5 (FastAPI) and refine through Module 10 (production gateway). No other GenAI course teaches this integrated pattern.


---

## Wrap-up

You've walked through all 7 code examples from this lesson. Re-run any cell to experiment — change a prompt, swap a model, raise the temperature. The published lesson page (linked at the top) has the surrounding narrative, quizzes, and practice exercises if you want to go deeper.
